# Siteora Image Model — Dataset Preparation

Build, validate, and inspect the image/caption dataset used for LoRA training.

In [ ]:
import sys
sys.path.insert(0, 'src')


## 0. (Optional) Scrape a starter dataset from Pexels

This pulls commercially-licensed stock photos via the [Pexels API](https://www.pexels.com/api/) instead of scraping arbitrary websites — Pexels' license explicitly allows free commercial use and modification (verified 2026-09-26, see `MODEL_LICENSE.md`/`dataset/README.md`), which matters because these images become training data for a commercial LoRA. A generic web scraper would pull in images of unknown/likely-prohibited license, so that was deliberately not built.

Get a free API key at https://www.pexels.com/api/, then either set it as a Colab secret named `PEXELS_API_KEY` (recommended — click the key icon in the left sidebar) or paste it directly below.

In [ ]:
import os
# Recommended: use a Colab secret instead of pasting your key in a cell.
# from google.colab import userdata
# os.environ['PEXELS_API_KEY'] = userdata.get('PEXELS_API_KEY')
os.environ['PEXELS_API_KEY'] = 'paste-your-key-here'


In [ ]:
from dataset.scraper import scrape_pexels, DEFAULT_QUERIES

n_added = scrape_pexels(
    queries=DEFAULT_QUERIES,   # covers SaaS/restaurant/real-estate/fashion/etc. — edit to add your own
    per_query=30,              # images per category
    dataset_dir='dataset',
)
print(f'Added {n_added} new images to dataset/metadata.jsonl')
print('Per-image source/license record: dataset/license_manifest.jsonl')


Captions written by the scraper are template-based (category + Pexels' own alt-text), not hand-reviewed. Look over a sample before training a real LoRA on them — Section 3 below (`review_captions`) does exactly that. Rewrite any captions that are vague or wrong; the scraper never overwrites captions you've edited.

## 1. Load images + captions

Expected layout: a directory of images plus a same-named `.txt` caption per image, OR an already-written `metadata.jsonl`. See `dataset/README.md`.

In [ ]:
from dataset.captions import generate_metadata_from_captions_dir, write_metadata

IMAGES_DIR = 'dataset/raw_images'  # your images + .txt captions
records = generate_metadata_from_captions_dir(IMAGES_DIR)
write_metadata(records, dataset_dir='dataset', metadata_file='metadata.jsonl')
print(f'wrote {len(records)} records')


## 2. Validate images and captions

In [ ]:
from dataset.validation import validate_dataset
report = validate_dataset('dataset')
print(report.summary())


## 3. Create metadata (done above) — inspect a few random pairs

In [ ]:
from dataset.captions import review_captions
for rec in review_captions('dataset', n=8):
    print(rec['file_name'], '->', rec['text'])


## 4. Display random image/caption pairs

In [ ]:
import json, random
from PIL import Image
import matplotlib.pyplot as plt

with open('dataset/metadata.jsonl') as f:
    records = [json.loads(l) for l in f if l.strip()]
sample = random.sample(records, min(6, len(records)))
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for ax, rec in zip(axes.flat, sample):
    img = Image.open(f"dataset/images/{rec['file_name']}")
    ax.imshow(img); ax.set_title(rec['text'][:40] + '...', fontsize=8); ax.axis('off')
plt.tight_layout(); plt.show()


## 5. Resize/crop to training resolution (writes to a SEPARATE directory)

In [ ]:
from dataset.preprocessing import preprocess_dataset
n = preprocess_dataset('dataset', 'dataset_processed_1024', target_size=1024)
print(f'processed {n} images into dataset_processed_1024/')


## 6. Create train/validation split

In [ ]:
from dataset.preprocessing import train_val_split
train_recs, val_recs = train_val_split('dataset_processed_1024', val_split=0.05)
print(len(train_recs), 'train /', len(val_recs), 'val')


## 7. Dataset statistics

In [ ]:
from dataset.preprocessing import dataset_statistics
print(dataset_statistics('dataset_processed_1024'))
